[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/15-projects/p10-finetune-lora-dpo/colab_quickstart.ipynb)

# P10 - Fine-tune a small model (LoRA SFT, then DPO): CPU quick start
Teaches SmolLM2-135M to turn support emails into strict JSON. This notebook runs a *quick mode* that finishes on a CPU in a few minutes; the full GPU recipe with TRL is in `colab_gpu_trl-colab-only.ipynb`.

## 1. Get the code
In Colab this clones the course repo. If you are already inside the project folder (running locally), the clone is skipped.

In [1]:
import os, pathlib, subprocess, sys
PROJECT = "p10-finetune-lora-dpo"
if pathlib.Path("src").is_dir() and pathlib.Path("README.md").exists():
    print("Project folder found - using the local copy, no clone needed.")
else:
    if not pathlib.Path("AI-Engineer").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/vvvvvivekkk/AI-Engineer.git"], check=True)
    os.chdir(f"AI-Engineer/15-projects/{PROJECT}")
print("working in", os.getcwd())

Project folder found - using the local copy, no clone needed.
working in /home/claude/AI-Engineer/15-projects/p10-finetune-lora-dpo


## 2. Install the pinned packages (Colab only)
Colab already has PyTorch with GPU support, so we skip the `torch` line to avoid replacing it. Locally you install with `pip install -r requirements.txt` in a venv.

In [2]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    reqs = [l for l in open("requirements.txt") if l.strip() and not l.startswith(("torch==", "#"))]
    open("/tmp/req.txt", "w").writelines(reqs)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/tmp/req.txt"], check=True)
print("ready")

ready


## 3. Build the datasets
1,500 email -> JSON pairs and 300 chosen/rejected pairs.

In [3]:
!python -m src.make_data

SFT: 1350 train / 150 test | DPO: 260 train / 40 test pairs


## 4. Step 1 - LoRA SFT (quick mode)
First measures the base model with plain prompting and with one example in the prompt, then trains LoRA adapters and measures again.

In [4]:
!python -m src.sft_lora_cpu --steps 15 --eval-n 5 --results results_sft_quick.json

Loading weights:   0%|                                 | 0/272 [00:00<?, ?it/s]

Loading weights:  50%|██████████▍          | 135/272 [00:00<00:00, 1344.72it/s]

Loading weights: 100%|█████████████████████| 272/272 [00:00<00:00, 1121.47it/s]


BEFORE (prompting)  zero-shot None
                    few-shot  {'valid_json': 1.0, 'exact_match': 0.0, 'order_id': 0.6, 'product': 0.4, 'intent': 0.0, 'urgency': 0.8}   (87s)
  sample few-shot output: {"order_id": "ORD-55555", "product": "SmartKettle", "intent": "disconnected", "urgency": "medium"}


trainable params: 1,843,200 || all params: 136,358,208 || trainable%: 1.3517


step   1  loss 0.519  (20s)


step  10  loss 0.332  (78s)


AFTER LoRA SFT (no examples in prompt) {'valid_json': 1.0, 'exact_match': 0.4, 'order_id': 0.6, 'product': 1.0, 'intent': 0.8, 'urgency': 0.8}
 email: Dear support, | My SmartKettle 2 keeps disconnecting from Wi-Fi. | Regards, | Jack 
   -> {"order_id": "ORD-12345", "product": "SmartKettle 2", "intent": "technical_support", "urgency": "medium"}
 email: hey | Can I return the HomeCam 360 (ORD-51937) and get a refund? | Regards, | Ben 
   -> {"order_id": "ORD-51937", "product": "HomeCam 360", "intent": "refund", "urgency": "medium"}
 email: Dear support, | I want a refund for my SmartKettle 2. | Thank you, | Liam 
   -> {"order_id": "ORD-12345", "product": "SmartKettle 2", "intent": "refund", "urgency": "medium"}


saved adapter -> outputs/sft-lora-adapter, metrics -> data/results_sft.json


## 5. Step 2 - DPO with a hand-written loss (quick mode)

In [5]:
!python -m src.dpo_cpu --steps 6 --train-n 12 --eval-n 4 --results results_dpo_quick.json

Loading weights:   0%|                                 | 0/272 [00:00<?, ?it/s]

Loading weights:  54%|███████████▍         | 148/272 [00:00<00:00, 1454.29it/s]

Loading weights: 100%|█████████████████████| 272/272 [00:00<00:00, 1514.54it/s]
starting from the SFT adapter in outputs/sft-lora-adapter


reference log-probs ready (25s). Held-out pairs where chosen is already more likely: 93%


JSON check before DPO: {'valid_json': 1.0, 'exact_match': 0.25, 'order_id': 0.5, 'product': 1.0, 'intent': 0.75, 'urgency': 0.75}


step   1  dpo loss 0.695  mean margin -0.03  (40s)


step   5  dpo loss 0.455  mean margin +8.46  (63s)


held-out pairs where chosen is more likely: before 93% -> after 93%; implicit-reward accuracy 93%


JSON check after DPO: {'valid_json': 0.5, 'exact_match': 0.25, 'order_id': 0.5, 'product': 0.5, 'intent': 0.25, 'urgency': 0.5}


saved adapter -> outputs/dpo-lora-adapter, metrics -> data/results_dpo.json


## 6. Step 3 - merge the adapter
Creates `outputs/merged-model` and an Ollama `Modelfile`, and prints the GGUF conversion commands.

In [6]:
!python -m src.merge_export

Loading weights:   0%|                                 | 0/272 [00:00<?, ?it/s]

Loading weights:  39%|████████▎            | 107/272 [00:00<00:00, 1058.02it/s]

Loading weights:  79%|████████████████▌    | 215/272 [00:00<00:00, 1066.19it/s]

Loading weights: 100%|█████████████████████| 272/272 [00:00<00:00, 1038.00it/s]


Writing model shards:   0%|                              | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████████████████| 1/1 [00:00<00:00,  1.45it/s]


merged dpo-lora-adapter -> /home/claude/AI-Engineer/15-projects/p10-finetune-lora-dpo/outputs/merged-model (542 MB); wrote outputs/Modelfile

Next steps (on a machine with git, cmake/pip and Ollama):
  git clone https://github.com/ggml-org/llama.cpp
  pip install -r llama.cpp/requirements.txt
  python llama.cpp/convert_hf_to_gguf.py /home/claude/AI-Engineer/15-projects/p10-finetune-lora-dpo/outputs/merged-model --outfile outputs/ticket-extractor-q8_0.gguf --outtype q8_0
  cd outputs && ollama create ticket-extractor -f Modelfile
  ollama run ticket-extractor "Hi, my FitBand Pro (ORD-12345) arrived broken."


## 7. Run the tests

In [7]:
!python -m pytest -q tests

...

.

..                                                                   [100%]
6 passed in 4.18s
